# Road network simplification

This notebook demonstrates how to simplify road networks with `geoai.simplify_road_network`, a convenience wrapper around [`neatnet`](https://uscuni.org/neatnet/). The workflow is useful for post-processing road centerlines from OpenStreetMap or road vectors derived from deep learning segmentation masks.

The simplification collapses transportation-oriented geometries such as dual carriageways, roundabouts, and complex intersections into a more morphological centerline network.

## Install optional dependencies

`neatnet` and `osmnx` are optional network-processing dependencies. Uncomment the next cell if they are not installed in your environment.

In [ ]:
# %pip install -U "geoai-py[networks]"

## Imports

In [ ]:
import matplotlib.pyplot as plt
import osmnx as ox

import geoai
import leafmap

## Download a sample road network

For a reproducible, lightweight example, download a small drivable street network from OpenStreetMap and project it to a local CRS before simplification.

In [ ]:
place = "Piedmont, California, USA"

# Download a drivable street graph and project it to a local CRS.
graph = ox.graph_from_place(place, network_type="drive")
graph = ox.project_graph(graph)

# Convert the graph edges to a GeoDataFrame of road geometries.
roads = ox.graph_to_gdfs(
    ox.convert.to_undirected(graph),
    nodes=False,
    edges=True,
    node_geometry=False,
    fill_edge_geometry=True,
).reset_index(drop=True)

roads = roads[["geometry"]].copy()
roads.head()

## Visualize the original road network

In [ ]:
ax = roads.plot(figsize=(8, 8), linewidth=0.6)
ax.set_title("Original road network")
ax.set_axis_off()

## Simplify the road network

Call `geoai.simplify_road_network()` with a GeoDataFrame. You can also pass a vector file path, for example `geoai.simplify_road_network("roads.gpkg")`.

`neatnet` may add a `_status` column describing whether each output geometry is original, changed, or newly created.

In [ ]:
simplified = geoai.simplify_road_network(roads)
simplified.head()

## Compare original and simplified networks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

roads.plot(ax=axes[0], linewidth=0.6, color="tab:blue")
simplified.plot(ax=axes[1], linewidth=0.8, color="tab:red")

axes[0].set_title("Original")
axes[1].set_title("Simplified")
for ax in axes:
    ax.set_axis_off()
plt.tight_layout()

In [ ]:
m = leafmap.Map()
m.add_gdf(
    roads,
    layer_name="Original",
    style={"color": "blue", "weight": 2},
    zoom_to_layer=True,
)
m.add_gdf(
    simplified,
    layer_name="Simplified",
    style={"color": "red", "weight": 2},
    zoom_to_layer=True,
)
m

## Save the simplified network

Write the simplified network to any vector format supported by GeoPandas/Fiona. GeoPackage is a good default for preserving CRS and attributes.

In [ ]:
output = "piedmont_roads_simplified.gpkg"
geoai.simplify_road_network(roads, output=output)
output

## Use an exclusion mask (optional)

If building footprints or other protected areas are available, pass them as `exclusion_mask` to preserve real enclosed blocks that should not be collapsed.

In [ ]:
# buildings = ox.features_from_place(place, tags={"building": True}).to_crs(roads.crs)
# simplified_with_buildings = geoai.simplify_road_network(
#     roads,
#     exclusion_mask=buildings.geometry,
# )